In [1]:
import os
import re
import fitz  # PyMuPDF for PDF text extraction
import pandas as pd
from langdetect import detect


start_keyword = [
    r"S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "SUMMARY"
    r"P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "PROJECTS SUMMARY"
    r"\bSUMMARY\s{1}T\s{1}A\s{1}B\s{1}L\s{1}E\b",  # "SUMMARY TABLE"
    r"Summary\s{1}o\s{1}f\s{1}P\s{1}R\s{1}O\s{1}P\s{1}O\s{1}S\s{1}A\s{1}L",  # "Summary of Proposal"
    r"R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "REQUESTS SUMMARY"
    r"T\s{1}E\s{1}C\s{1}H\s{1}N\s{1}I\s{1}C\s{1}A\s{1}L\s{1}A\s{1}S\s{1}S\s{1}I\s{1}S\s{1}T\s{1}A\s{1}N\s{1}C\s{1}E\s{1}R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "TECHNICAL ASSISTANCE REQUESTS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROJECTS SUMMARY"
    r"\bE\s{1}X\s{1}E\s{1}C\s{1}U\s{1}T\s{1}I\s{1}V\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "EXECUTIVE SUMMARY"
    r"\bS\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\s{1}T\s{1}A\s{1}B\s{1}L\s{1}E\b",  # "SUMMARY TABLE"
    r"S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\s{1}O\s{1}F\s{1}P\s{1}R\s{1}O\s{1}P\s{1}O\s{1}S\s{1}A\s{1}L",  # "SUMMARY OF PROPOSAL"
    r"R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "REQUESTS SUMMARY"
    r"T\s{1}E\s{1}C\s{1}H\s{1}N\s{1}I\s{1}C\s{1}A\s{1}L\s{1}A\s{1}S\s{1}S\s{1}I\s{1}S\s{1}T\s{1}A\s{1}N\s{1}C\s{1}E\s{1}R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "TECHNICAL ASSISTANCE REQUESTS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROJECTS SUMMARY"
    r"\bE\s{1}X\s{1}E\s{1}C\s{1}U\s{1}T\s{1}I\s{1}V\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  
    r"\n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b",
    r"\n{1,}(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|Executive)\s*SUMMARY",
    r"^(?:\S+\s+){0,3}Summary(?:\s+\S+){0,3}$",  # Retain this if "Summary" as a heading is important
    r"(?i)SUMMARY TABLE",  # "SUMMARY TABLE"
    r"SUMMARY TABLE",
    r"\n{1,}\s*SUMMARY(\s{2,}|(\s*\n))",
    r"(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|STUDY|EXECUTIVE)\s*SUMMARY(\s{3,}|\n)",
    r"\s{2,}(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|STUDY|EXECUTIVE)\s*SUMMARY(\s{3,}|\n)",
    r"(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|Executive)\s*SUMMARY",
    r"(project’s|program|programm|programme|project|study|executive)\s*summary",
    r"PROJECT\s*SUMMARY(\s{3,}|(\s*\n))",
    r"\bPROGRAMME SUMMARY\b",  # "PROGRAMME SUMMARY"
    r"\bPROGRAM SUMMARY\b",  # "PROGRAM SUMMARY"
    r"\bPROGRAMM SUMMARY\b",  # "PROGRAMM SUMMARY"
    r"\bPROJECT SUMMARY\b",  # "PROJECT SUMMARY"
    r"\bEXECUTIVE SUMMARY\b",  # "EXECUTIVE SUMMARY"
    r"^(?:\S+\s+){0,3}Summary(?:\s+\S+){0,3}$",
    r"(?:\S+\s+){0,3}Summary(?:\s+\S+){0,3}$",
]

end_keyword = [
    r"F\s{1}R\s{1}A\s{1}M\s{1}E\s{1}W\s{1}O\s{1}R\s{1}K",
    r"\n{1,}\s*Results-based",
    r"\n{1,}\s*Result",
    r"\n{1,}\s*(Project’s|Program|Programme|Programm)\s*Results-Based",
    r"Framework(\s{3,}|(\s*\n))",
    r"LIST OF ACRONYMS",  # "LIST OF ACRONYMS"
    r"ACRONYMS",  # "ACRONYMS"
    r"T\s*a\s*b\s*l\s*e\s*\s*o\s*f\s*\s*C\s*o\s*n\s*t\s*e\s*n\s*t",
    r"\bTable of Contents\b",  # "Table of Contents"
    r"T\s{1}A\s{1}B\s{1}L\s{1}E",
    r"F\s{1}R\s{1}A\s{1}M\s{1}E\s{1}W\s{1}O\s{1}R\s{1}K",  # "FRAMEWORK"
    r"R\s{1}E\s{1}S\s{1}U\s{1}L\s{1}T",  # "RESULT"
    r"L\s{1}O\s{1}G\s{1}I\s{1}C\s{1}A\s{1}L",  # "LOGICAL"
    r"\n{1,}\s*RESULTS\s*FRAMEWORK",
    r"RESULTS\s*FRAMEWORK(\s{3,}|\n)",
    r"\n{1,}\s*RESULTS-FRAMEWORK",
    r"RESULTS-FRAMEWORK(\s{3,}|\n)",
    r"\n{1,}\s*LOGICAL-FRAMEWORK",
    r"LOGICAL-FRAMEWORK(\s{3,}|\n)",
    r"\n{1,}\s*LOGICAL\s*FRAMEWORK",
    r"LOGICAL\s*FRAMEWORK(\s{3,}|\n)",
    r"\n{1,}\s*RESULT\s*FRAMEWORK",
    r"RESULT-BASED\s*FRAMEWORK(\s{3,}|\n)",
    r"\n{1,}\s*RESULTS-BASED\s*FRAMEWORK",
    r"RESULTS-BASED\s*FRAMEWORK(\s{3,}|\n)",
    r"RESULTS\s*MATRIX(\s{3,}|\n)",
    r"\n{1,}\s*RESULTS\s*MATRIX",
    r"RESULT\s*FRAMEWORK(\s{3,}|\n)",
    r"\n{1,}\s*RESULT\s*MATRIX(\s{3,}|\n)",
    r"\n{1,}\s*POLICY\s*MATRIX",
    r"POLICY\s*MATRIX(\s{3,}|\n)",
    r"RESULTS\s*FRAMEWORK\s*FOR\s*THE\s*IRSUE-LUXOR\s*PROGRAM",
    r"RESULTS\s*FRAMEWORK",  # Matches "RESULT FRAMEWORK"
    r"\bLOGICAL\s*FRAMEWORK\b",  # "LOGICAL FRAMEWORK"
    r"\bResult-based\s*Framework\b",  # "Result-based Framework"
    r"\bResults-based\s*Framework\b",  # "Results-based Framework"
    r"RESULTS FRAMEWORK"# "Result based",
    r"^(?:\S+\s+){0,4}FRAMEWORK(?:\s+\S+){0,4}$",
    r"(?:\S+\s+){0,4}FRAMEWORK(?:\s+\S+){0,4}$",
    r"^(?:\S+\s+){0,4}MATRIX(?:\s+\S+){0,4}$",
    r"(?:\S+\s+){0,4}MATRIX(?:\s+\S+){0,4}$",
    r"\n{1,}I\.\s*INTRODUCTION",
    r"\d{1,2}\.\d?\s*RESULT\s*FRAMEWORK", # Matches "RESULT FRAMEWORK"
    r"\d{1,2}\.\d?\s*LOGICAL\s*FRAMEWORK",  # Matches "LOGICAL FRAMEWORK"
    r"\d{1,2}\.\d?\s*RESULTS\s*FRAMEWORK", #
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(RESULT|RESULTS|LOGICAL|Result-based)\s*FRAMEWORK(\s{3,}|\n)",
    r"\b[BCDEF]\s*[-\.)]?\s*\n?\s*(RESULT|RESULTS|LOGICAL|Result-based)\s*FRAMEWORK(\s{3,}|\n)",
    r"\d{1,2}(?:\.\d{1,2})?(?:\.\d{1,2})?\s*(\.|-)?\s*\n?\s*(\b\w+\b\s*){0,4}(RESULT|RESULTS|LOGICAL|Result-based)\s*FRAMEWORK(\b\w+\b\s*){0,4}",
    r"\b[BCDEF]\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}(RESULT|RESULTS|LOGICAL|Result-based)\s*FRAMEWORK(\b\w+\b\s*){0,4}",
    r"Project’s Results-Based Logical Framework"
       
]

start_patterns = [
    
    r"\d{1}\s*[-)\.]\s*n?\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)\s*DESCRIPTION",
    r" D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N\s{1}O\s{1}F\s{1}T\s{1}H\s{1}E\s{1}S\s{1}T\s{1}U\s{1}D\s{1}I\s{1}E\s{1}S ",
    r" P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N\s{1}A\s{1}N\s{1}D\s{1}F\s{1}I\s{1}N\s{1}A\s{1}N\s{1}C\s{1}I\s{1}N\s{1}G ",
    r" P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N ",
    r" P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E\s{1}D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N",
    r" P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N ",
    r" P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N ",
    r" S\s{1}T\s{1}U\s{1}D\s{1}Y\s{1}D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T\s{1}I\s{1}O\s{1}N ",
    r" P\s{1}R\s{1}O\s{1}P\s{1}O\s{1}S\s{1}E\s{1}D\s{1}P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E ",
    r" P\s{1}R\s{1}O\s{1}P\s{1}O\s{1}S\s{1}E\s{1}D\s{1}P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M ",
    r" T\s{1}H\s{1}E\s{1}P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E ",
    r" D\s{1}E\s{1}S\s{1}C\s{1}R\s{1}I\s{1}P\s{1}T",
    r"\d{1}\s*[-)\.]\s*n?\s*Detailed\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*Detailed\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"\b[A-F]\s*[-\.)]?\s*\n?\s*Detailed\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Detailed\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"\d{1}\s*[-)\.]\s*n?\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY|activity|STUDIES)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY|activity|STUDIES)",
    r"\b[A-F]\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY|activity|STUDIES)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*OF\s*THE\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY|activity|STUDIES)",
    r"\d{1}\s*[-)\.]\s*n?\s*DESCRIPTION\s*OF\s*THE\s*PROPOSED\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*DESCRIPTION\s*OF\s*THE\s*PROPOSED\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"\b[A-F]\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*OF\s*THE\s*PROPOSED\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*OF\s*THE\s*PROPOSED\s*(operation|Project’s|PROGRAM|PROGRAMM|PROGRAMME|projects|Project|STUDY|activity)",
    r"\d{1}\s*[-)\.]\s*n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*Description",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*Description",
    r"\b[A-F]\s*[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*Description",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*Description",
    r"\d{1}\s*[-)\.]\s*n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*DESCRIPTION\s*AND\s*FINANCING",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*DESCRIPTION\s*AND\s*FINANCING",
    r"\b[A-F]\s*[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*DESCRIPTION\s*AND\s*FINANCING",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*DESCRIPTION\s*AND\s*FINANCING",
    r"\d{1}\s*[-)\.]\s*n?\s*(PROPOSED|The)\s*(operation|activity|Project’s|PROGRAM|projects|PROGRAMM|PROGRAMME|Project|STUDY)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(PROPOSED|The)\s*(operation|activity|Project’s|PROGRAM|projects|PROGRAMM|PROGRAMME|Project|STUDY)",
    r"\b[A-F]\s*[-\.)]?\s*\n?\s*(PROPOSED|The)\s*(operation|activity|Project’s|PROGRAM|projects|PROGRAMM|PROGRAMME|Project|STUDY)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(PROPOSED|The)\s*(operation|activity|Project’s|PROGRAM|projects|PROGRAMM|PROGRAMME|Project|STUDY)",
    r"\bSECTION\s*2:\s*PROJECT\s*DETAILS\b",
    r"\b3\s*2021\s*PROGRAMME\s*\–?\s*PAMRER\s*II\b",
    r"3 2021 PROGRAMME – PAMRER II",
    r"d\.\s*\n?\s*THE\*PROJECT",
    r"2021 PROGRAMME – PAMRER II",
    r"\d\s*\n?\s*2021\s*PROGRAMME\s*\–\s*PAMRER\s*II"
]


stop_patterns = [
    r"F\s{1}E\s{1}A\s{1}S\s{1}I\s{1}B\s{1}I\s{1}L\s{1}I\s{1}T\s{1}Y",
    r"P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}F\s{1}E\s{1}A\s{1}S\s{1}I\s{1}B\s{1}I\s{1}L\s{1}I\s{1}T\s{1}Y ",
    r"P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}F\s{1}E\s{1}A\s{1}S\s{1}I\s{1}B\s{1}I\s{1}L\s{1}I\s{1}T\s{1}Y ",
    r"P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E\s{1}I\s{1}M\s{1}P\s{1}L\s{1}E\s{1}M\s{1}E\s{1}N\s{1}T\s{1}A\s{1}T\s{1}I\s{1}O\s{1}N ",
    r"P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}I\s{1}M\s{1}P\s{1}L\s{1}E\s{1}M\s{1}E\s{1}N\s{1}T\s{1}A\s{1}T\s{1}I\s{1}O\s{1}N ",
    r"P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}I\s{1}M\s{1}P\s{1}L\s{1}E\s{1}M\s{1}E\s{1}N\s{1}T\s{1}A\s{1}T\s{1}I\s{1}O\s{1}N ",
    r"P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}I\s{1}M\s{1}P\s{1}L\s{1}E\s{1}M\s{1}E\s{1}N\s{1}T\s{1}A\s{1}T\s{1}I\s{1}O\s{1}N ",
    r"E\s{1}S\s{1}T\s{1}I\s{1}M\s{1}A\s{1}T\s{1}E\s{1}D\s{1}C\s{1}O\s{1}S\s{1}T\s{1}F\s{1}O\s{1}R\s{1}T\s{1}H\s{1}E\s{1}P\s{1}R\s{1}E\s{1}P\s{1}A\s{1}R\s{1}A\s{1}T\s{1}O\s{1}R\s{1}Y\s{1}A\s{1}C\s{1}T\s{1}I\s{1}V\s{1}I\s{1}T\s{1}I\s{1}E\s{1}S ",
    r"C\s{1}O\s{1}S\s{1}T\s{1}E\s{1}S\s{1}T\s{1}I\s{1}M\s{1}A\s{1}T\s{1}E\s{1}S\s{1}F\s{1}O\s{1}R\s{1}T\s{1}H\s{1}E\s{1}P\s{1}R\s{1}E\s{1}P\s{1}A\s{1}R\s{1}A\s{1}T\s{1}O\s{1}R\s{1}Y\s{1}A\s{1}C\s{1}T\s{1}I\s{1}V\s{1}I\s{1}T\s{1}I\s{1}E\s{1}S ",
    r"I\s{1}M\s{1}P\s{1}L\s{1}E\s{1}M\s{1}E\s{1}N\s{1}T\s{1}A\s{1}T\s{1}I\s{1}O\s{1}N",
    r"\d{1}\s*[-)\.]\s*n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|OPERATIONAL|OPERATION)\s*IMPLEMENTATION",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|OPERATIONAL|OPERATION)\s*IMPLEMENTATION", #### Perfect pattern for numbers
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|OPERATIONAL|OPERATION)\s*IMPLEMENTATION",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|OPERATIONAL|OPERATION)\s*IMPLEMENTATION",
    r"\d{1}\s*[-)\.]\s*n?\s*IMPLEMENTATION(\s{3,}|\n)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*IMPLEMENTATION(\s{3,}|\n)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*IMPLEMENTATION(\s{3,}|\n)",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*IMPLEMENTATION(\s{3,}|\n)",
    r"\d{1}\s*[-)\.]\s*n?\s*Feasibility(\s{3,}|\n)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*Feasibility(\s{3,}|\n)",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Feasibility(\s{3,}|\n)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Feasibility(\s{3,}|\n)",
    r"\d{1}\s*[-)\.]\s*n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*FEASIBILITY",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)\s*FEASIBILITY",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)\s*FEASIBILITY",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)\s*FEASIBILITY",
    r"\d{1}\s*[-)\.]\s*n?\s*(COST|COSTS)\s*ESTIMATES\s*OF\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(COST|COSTS)\s*ESTIMATES\s*OF\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(COST|COSTS)\s*ESTIMATES\s*OF\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(COST|COSTS)\s*ESTIMATES\s*OF\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"\d{1}\s*[-)\.]\s*n?\s*ESTIMATED\s+(COST|COSTS)\s+OF\s+THE\s+(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*ESTIMATED\s+(COST|COSTS)\s+OF\s+THE\s+(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*ESTIMATED\s+(COST|COSTS)\s+OF\s+THE\s+(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*ESTIMATED\s+(COST|COSTS)\s+OF\s+THE\s+(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY)",
    r"\d{1}\s*[-)\.]\s*n?\s*IMPLEMENTATION,\s*MONITORING\s*AND\s*EVALUATION",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*IMPLEMENTATION,\s*MONITORING\s*AND\s*EVALUATION",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*IMPLEMENTATION,\s*MONITORING\s*AND\s*EVALUATION",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*IMPLEMENTATION,\s*MONITORING\s*AND\s*EVALUATION",
    r"\d{1}\s*[-)\.]\s*n?\s*IMPLEMENTATION\s*AND\s*EVALUATION",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*IMPLEMENTATION\s*AND\s*EVALUATION",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*IMPLEMENTATION\s*AND\s*EVALUATION",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*IMPLEMENTATION\s*AND\s*EVALUATION",
    r"\d{1}\s*[-)\.]\s*n?\s*Implementation\s*(Arrangements|Arrangement)",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*Implementation\s*(Arrangements|Arrangement)",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Implementation\s*(Arrangements|Arrangement)",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Implementation\s*(Arrangements|Arrangement)",
    r"\d{1}\s*[-)\.]\s*n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*ESTIMATED\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*COST\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\d{1}\s*[-)\.]\s*n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*ESTIMATED\s*COST\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\d{1}\s*[-)\.]\s*n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\d{1}\s*[-)\.]\s*n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*PREPARATORY\s*ACTIVITIES",
    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*COST\s*ESTIMATES\s*FOR\s*THE\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Operational|Operation|STUDY|activity|projects)\s*PREPARATORY\s*ACTIVITIES",
    r"3. COST ESTIMATES OF THE PROJECT (IN UA)",
    r"\bIII\.\s*The\s*New\s*Proposal"

]

In [2]:
import signal

# Define a custom exception for timeout
class TimeoutException(Exception):
    pass

# Timeout handler
def timeout_handler(signum, frame):
    raise TimeoutException()

In [3]:
def is_pdf_in_english(text):
    try:
        return detect(text) == 'en'
    except Exception as e:
        print(f"Error detecting language: {e}")
        return False

        

def clean_text(text, cleaning_patterns):
    # List of regex patterns to search for
    
    for pattern in cleaning_patterns:
        # Search for the pattern in the text
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Print the pattern that is used to truncate the text
            print(f"Pattern used to truncate text:\n\n----------- {pattern}")
            
            # Truncate the text after the matched pattern
            text = text[:match.start()]
    
    return text
        


def remove_illegal_characters(df):
    # Define a function to clean individual cells
    def clean_cell(value):
        if isinstance(value, str):
            # Remove illegal characters but preserve line spaces, tabs, and carriage returns
            return re.sub(r'[^\x09\x0A\x0D\x20-\x7E]', '', value)
        return value

    # Apply the cleaning function to the entire DataFrame
    return df.applymap(clean_cell)

def identify_table_of_contents(pdf):
    """
    Identifies the Table of Contents (TOC) page based on specific patterns.
    Returns the human-readable page number (1-indexed), or 1 (default to first page) if not found.
    """
    # Iterate over each page in the PDF
    for page_num in range(pdf.page_count):
        page = pdf[page_num]
        page_text = page.get_text("text")
        
        # List of patterns to match variations of "Table of Contents" and "Contents"
        toc_patterns = [
            r"T\s{1,}a\s{1,}b\s{1,}l\s{1,}e\s{1,}o\s{1,}f\s{1,}\s{1,}C\s{1,}o\s{1,}n\s{1,}t\s{1,}e\s{1,}n\s{1,}t\s{1,}s",  # 'T a b l e o f C o n t e n t s'
            r"C\s{1,}o\s{1,}n\s{1,}t\s{1,}e\s{1,}n\s{1,}t\s{1,}s",  # 'C o n t e n t s'
            r"T A B L E O F C O N T E N T S",  # 'T A B L E O F C O N T E N T S'
            r"Table\s*of\s*Contents",  # 'Table of Contents'
            r"Contents"  # 'Contents'
        ]
        
        # Compile the combined pattern using OR (|) for alternation
        toc_pattern = re.compile("|".join(toc_patterns), re.IGNORECASE)

        # Search for any of the patterns in the page text
        if toc_pattern.search(page_text):
            return page_num + 1  # Return as human-readable page number
    
    # Default to the first page if no TOC is found
    return 1


def extract_summary_section(file_path):
    """
    Extracts the summary section from the PDF based on start and end keywords,
    skipping only the Table of Contents (TOC) page and limiting the extracted content to two pages.
    """
    try:
        pdf_document = fitz.open(file_path)

        # Identify the TOC page
        toc_page = identify_table_of_contents(pdf_document) - 1  # Convert to 0-indexed
        print(f"Table of Contents identified on page: {toc_page + 1}")

        # Combine all pages into a single string, skipping only the TOC page
        filtered_pages = []
        for page_num in range(pdf_document.page_count):
            if page_num == toc_page:  # Skip only the TOC page
                continue
            page_text = pdf_document[page_num].get_text("text")
            filtered_pages.append(page_text)

        filtered_content = "\f".join(filtered_pages)

        # Find the start match
        start_index, matched_start_pattern, start_page_index, start_line = None, None, None, None
        for pattern in start_keyword:
            match = re.search(pattern, filtered_content, re.DOTALL | re.IGNORECASE)
            if match:
                start_index = match.end()
                matched_start_pattern = pattern
                try:
                    start_page_index = next(
                        (i for i, page in enumerate(filtered_pages) if match.group(0) in page), None
                    )
                except StopIteration:
                    start_page_index = None

                if start_page_index is not None:
                    page_lines = filtered_pages[start_page_index].splitlines()
                    for line in page_lines:
                        if re.search(pattern, line, re.IGNORECASE):
                            start_line = line.strip()
                            break

                print(f"Matched start pattern: {matched_start_pattern} on page {start_page_index + 1 if start_page_index is not None else 'Unknown'}")
                print(f"Actual line containing start pattern: {start_line}")
                break

        if start_index is None:
            print("Start Summary Table not found.")
            pdf_document.close()
            return "", ""

        # Find the end match
        end_index, matched_end_pattern = None, None
        for pattern in end_keyword:
            match = re.search(pattern, filtered_content[start_index:], re.DOTALL | re.IGNORECASE)
            if match:
                end_index = start_index + match.start()
                matched_end_pattern = pattern
                try:
                    end_page_index = next(
                        (i for i, page in enumerate(filtered_pages) if match.group(0) in page), None
                    )
                except StopIteration:
                    end_page_index = None

                print(f"Matched end pattern: {matched_end_pattern} on page {end_page_index + 1 if end_page_index is not None else 'Unknown'}")
                break

        # Limit the extracted section to two pages
        if start_page_index is not None:
            two_page_limit_index = len("\f".join(filtered_pages[: start_page_index + 3]))
            end_index = min(end_index or len(filtered_content), two_page_limit_index)

        pdf_document.close()

        if start_index is not None and end_index is not None:
            print("Summary Table found!!!")
            return filtered_content[start_index:end_index].strip(), "Summary Section"
        else:
            print("Could not determine the end of the Summary Table.")
            return "", ""

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Error processing the PDF", ""


        

def extract_description(pdf_path):
    """
    Reads text from a PDF between specified start and stop patterns.
    Handles extensive regex patterns efficiently and prints matched patterns.
    """
    try:
        with fitz.open(pdf_path) as pdf_document:
            # Pre-compile regex patterns
            compiled_start_patterns = [re.compile(pattern, re.DOTALL | re.IGNORECASE) for pattern in start_patterns]
            compiled_stop_patterns = [re.compile(pattern, re.DOTALL | re.IGNORECASE) for pattern in stop_patterns]

            # Identify Table of Contents (TOC) page
            toc_page = identify_table_of_contents(pdf_document) or 1
            print(f"TOC identified on page {toc_page}")

            # Initialize variables
            text_chunks = []
            start_found, extracted_text = False, None
            matched_start_pattern = None
            matched_stop_pattern = None

            # Search for start pattern
            for page_num in range(toc_page + 1, pdf_document.page_count + 1):
                page = pdf_document[page_num - 1]
                page_text = page.get_text("text")
                for pattern in compiled_start_patterns:
                    match = pattern.search(page_text)
                    if match:
                        start_found = True
                        matched_start_pattern = pattern.pattern  # Capture the matched pattern
                        start_index = match.end()
                        text_chunks.append(page_text[start_index:])
                        print(f"Description Start pattern matched: '{matched_start_pattern}' on page {page_num}")
                        break
                if start_found:
                    break

            if not start_found:
                print("Start pattern not found.")
                return "Pattern Not Found", "N/A"

            # Search for stop pattern
            for page_num in range(page_num + 1, pdf_document.page_count + 1):
                page = pdf_document[page_num - 1]
                page_text = page.get_text("text")
                text_chunks.append(page_text)
                for pattern in compiled_stop_patterns:
                    match = pattern.search("".join(text_chunks))
                    if match:
                        matched_stop_pattern = pattern.pattern  # Capture the matched pattern
                        extracted_text = "".join(text_chunks)[: match.start()]
                        print(f"Description Stop pattern matched: '{matched_stop_pattern}' on page {page_num}")
                        break
                if extracted_text:
                    break

            # Return results
            if not extracted_text:
                print("Stop pattern not found. Returning text after start pattern.")
                return "".join(text_chunks).strip(), "N/A"

            # Return the matched patterns with the extracted description
            #print(f"Matched Start Pattern: {matched_start_pattern}")
            #print(f"Matched Stop Pattern: {matched_stop_pattern}")
            return extracted_text.strip(), "Description Section"

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Error processing the PDF", ""




def process_pdfs_in_folder(folder_path, start_patterns, stop_patterns, output_excel):
    results = []

    

    for filename in os.listdir(folder_path):
        if filename.endswith('.pdf'):
            file_path = os.path.join(folder_path, filename)
            print(f"\nProcessing file: {filename}")
            try:
                pdf_document = fitz.open(file_path)

                # Extract all text for language check
                text = ""
                for page_num in range(pdf_document.page_count):
                    text += pdf_document[page_num].get_text("text")

                # Check language first
                if not is_pdf_in_english(text):
                    results.append([filename, "Not Supported", "Not Supported"])
                    pdf_document.close()
                    continue

                # Identify the Table of Contents page
                toc_page = identify_table_of_contents(pdf_document)
                if toc_page is not None:
                    print(f"Table of Contents identified on page: {toc_page + 1}")
                    skip_pages = list(range(toc_page + 2))  # Skip TOC page and all pages before it
                else:
                    skip_pages = []

                # Extract text excluding TOC pages (i.e., start from the page after the TOC)
                text = ""
                for page_num in range(pdf_document.page_count):
                    if page_num in skip_pages:  # Skip TOC pages and earlier ones
                        continue
                    text += pdf_document[page_num].get_text("text")
                pdf_document.close()
                
                project_component_start = [
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Objectives\s*and\s*Components([\s\S]*)",#126
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|Projects|study|activity|operational|operation)\s*Objectives\s*and\s*Components([\s\S]*)",#126
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)",#553
                    r"\d{1}\s*[-)\.]\s*n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|Projects|study|activity|operational|operation)\s*Objectives\s*and\s*Components([\s\S]*)",
                    r"\d{1}\s*[-)\.]\s*n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|Projects|study|activity|operational|operation)\s*Components([\s\S]*)",#553
                    r"C\s{1}o\s{1}m\s{1}p\s{1}o\s{1}n\s{1}e\s{1}n\s{1}t\s{1}s([\s\S]*)",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)", ### new
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Objectives\s*and\s*Components([\s\S]*)",
                    r"\b[ABCDEFG]\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)", #143
                    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Projects|Project)\s*Components([\s\S]*)", #new
                    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Projects|Project)\s*Component([\s\S]*)", #new
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Projects|Project)\s*Components([\s\S]*)", #new
                    r"((\b[B-F])|(\b[IVIX]{1,3})|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Projects|Project)\s*Components([\s\S]*)", #new
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(main|overall)?\s*(Project|Program|Programme|Programm|project's)?\s*Components([\s\S]*)",#55
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Description\s*of\s*(Project|Program|Programme|Programm|project's)\s*components([\s\S]*)",#19
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Description\s*of\s*(Project|Program|Programme|Programm|project's)\s*components([\s\S]*)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Overall\s*Scope\s*and\s*Components([\s\S]*)",#14
                    r"\d{1}\s*[-)\.]\s*n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Projects|Project)\s*Components([\s\S]*)", #new
                    r"\d{1}\s*[-)\.]\s*n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Projects|Project)\s*Components([\s\S]*)", #new
                    r"\d{1}\s*[-)\.]\s*n?\s*(main|overall)?\s*(Project|Program|Programme|Programm|project's)?\s*Components([\s\S]*)",#55
                    r"\d{1}\s*[-)\.]\s*n?\s*Description\s*of\s*(Project|Program|Programme|Programm|project's)\s*components([\s\S]*)",#19
                    r"\d{1}\s*[-)\.]\s*n?\s*(Project|Program|Programme|Programm|project's)\s*components([\s\S]*)",
                    r"\d{1}\s*[-)\.]\s*n?\s*Overall\s*Scope\s*and\s*Components([\s\S]*)",#14
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Component\s*(1|one)([\s\S]*)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Component\s*(1|one)([\s\S]*)", #13
                    r"((\d{1}(\.)?(\d)?)|([ABCD]))\.\s*(\n)?\s*(Project|Program|Programm|Programme)\s*Components([\s\S]*)", #3
                    r"\b[ABCDEFG]\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}Components(\b\w+\b\s*){0,4}([\s\S]*)", #220
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(\b\w+\b\s*){0,4}Components(\b\w+\b\s*){0,4}([\s\S]*)", #103
                    r"(\b[A-D]\s*(\.|-)?)\s*\n?\s*(\b\w+\b\s*){0,4}Components(\b\w+\b\s*){0,4}([\s\S]*)",
                    r"\b(I{1,3}|IV|V|VI{0,3}|VII{0,3}|VIII{0,3})\b\s*(\.|-)?\s*\n?\s*(\b\w+\b\s*){0,4}Components(\s+\b\w+\b\s*){0,4}([\s\S]*)",#8
                    r"three\s*components([\s\S]*)", #16
                    r"two\s*components([\s\S]*)", 
                    r"four\s*components([\s\S]*)",
                    r"five\s*components([\s\S]*)", 
                    r"Component\s*1\s+([\s\S]*)",#9
                    r"Component\s*1\s*([\s\S]*)",#9
                    r"major\s*components([\s\S]*)", ### 2
                    r"components:([\s\S]*)",#20
                    r"main\s*components([\s\S]*)", #4
                    r"major\s*components([\s\S]*)", ### 2
                    r"key\s*components([\s\S]*)", ### 2
                    r"primary\s*components([\s\S]*)", ### 2
                    r"Component\sName([\s\S]*)", #4
                    r"Component\s+I\s+([\s\S]*)", #2
                    r"Component\s+A\s+([\s\S]*)", #2
                    r"components,\s+([\s\S]*)", #4
                    r"(Project|Programme|Programm|Program)\s*Components([\s\S]*)", #7
                    r"(Project|Programme|Programm|Program)\s*Components\s*:([\s\S]*)", #7
                    r"\b[ABCDEFG]\s*[-\.]?\s*\n?\s*Component\s*(1|one)([\s\S]*)", #1
                    r"Programme\s*Components:([\s\S]*)",
                      # Handles spacing variations with "PROGRAMME COMPONENTS"
                    
                ]
                
                project_component_end = [ #Improving list
                    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*feasibility",
                    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*Feasibility",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*cost\s*and\s*financing\s*arrangements",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*Area\s*Beneficiaries",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Implementation",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Risks",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project|Projects|Project's)\s*(COST|COSTS)",
                    r"\b[IVIX]{1,3}[-\.)]?\s*\n?\s*Implementation",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*feasibility",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Feasibility",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project's|Projects)\s*target\s*area\s*and\s*population",                  
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Participatory\s*process\s*for\s*project\s*identification,\s*design\s*and\s*implementation",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Bank\s*Group\s*experience,\s*lessons reflected in project design",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Key\s*performance\s*indicators",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Project\s*Assumptions\s*and\s*Risks",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*Implementation(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*Beneficiaries",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*Costs",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*Costs(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Sustainability|Governance|Risk\s*management|Knowledge\s*Building|Legal\s*instrument)(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Environmental\s*and\s*Social\s*Impact(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*(Estimated\s*Costs)|(Target\s*Area\s*and)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}Technical\s*(Solutions|Solution)(\b\w+\b\s*){0,4}",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Risks(\s{3,}|\n)",                  
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(PROJECT|PROGRAM|PROGRAMM|PROGRAMME)\s*TYPE(\s{3,}|\n)",                    
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Policy\s*Dialogue(\s{3,}|\n)",              
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Target\s*Beneficiaries(\s{3,}|\n)",                   
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\sLoan\s*Conditions(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*implementation(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(RESULT|RESULTS|LOGICAL|Result-based)\s*FRAMEWORK(\s{3,}|\n)",     
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Technical\s*(Solutions|Solution)\s*Retained\s*and\s*Other\s*Alternatives\s*Explored(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Budget(\s{3,}|\n)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Projects)\s*Beneficiaries",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Project\s*(Costs|Cost)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Project\s*(Costs|Cost)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*PROJECT\s*TARGET\s*AREA",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*BANK\s*EXPERIENCE\s*AND",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*(Costs|Cost)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Key\s*performance\s*indicators",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Participatory\s*process\s*for\s*project\s*identification,\s*design\s*and\s*implementation",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Environmental\s*and\s*Social\s*Impact",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Sustainability|Governance|Risk\s*management|Knowledge\s*Building|Legal\s*instrument)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*feasibility",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Implementation(\s{3,}|\n)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Risks",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Technical\s*(Solutions|Solution)\s*Retained\s*and\s*Other\s*Alternatives",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Technical\s*(Solutions|Solution)\s*explored\s*and\s*Alternatives\s*Considered",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Loan\s*Conditions",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Policy\s*Dialogue",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Result-based\s*Framework",  # Matches "Result-based Framework"
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Results-based\s*Framework",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Description\s*of\s*the\s*expected\s*outputs",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Cost\s*Estimates\s*of\s*the\s*project",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Outputs and Key activities ",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project’s Estimated Costs",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Technical\s*Solution\s*Adopted\s*and\s*Alternatives\s*Explored", 
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*ESTIMATED COSTS OF PREPARATORY ACTIVITIES",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*Cost\s*and\s*Financing\s*Arrangements",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Anticipated\s*Economic\s*and\s*Social\s*Benefits",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*implementation\s*arrangements", 
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Target\s*Beneficiaries",  # Matches "Target Beneficiaries"
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Background\s*and\s*Justification\s*of\s*the\s*Request",  # Matches "Background and Justification of the Request"       
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Institutional\s*and\s*Implementation\s*Arrangements ",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Expected\s*Project\s*Outcomes\s*and\s*Impacts",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*cost\s*and\s*funding\s*arrangements",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*cost\s*by\s*category\s*of\s*expenditures ",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Lessons\s*learned\s*related\s*to\s*efficiency",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project’s\s*Target\s*Area\s*and\s*Population\s*Beneficiaries",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Lessons\s*Reflected",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*cost\s*and\s*funding\s*arrangements",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Expected\s*Outputs",  # Matches "Expected Outputs"
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Programme\s*Beneficiaries",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Project\s*Target\s*Area,\s*Beneficiaries\s*and\s*Other\s*Stakeholders",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Technical\s*Solutions\s*Retained\s*and\s*Other\s*Alternatives\s*Explored",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Description\s*of\s*expected\s*outputs",  # Matches "Description of expected outputs"
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Description\s*of\s*the\s*expected\s*outputs",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Cost\s*Estimates\s*of\s*the\s*project",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Theory\*of\s*Change",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*IMPLEMENTATION,\s*MONITORING\s*AND\s*EVALUATION ",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Technical\s*Options\s*and\s*Alternative\s*Options\s*Considered",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Costing\s*and\s*Financial\s*Arrangements",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Expected\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*Outcomes\s*and\s*Impacts(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Institutional\s*and\s*Implementation\s*Arrangements(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*implementation\s*(Schedule|(AND\s*EVALUATION))(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*implementation\s*(arrangements|arrangement)(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Background\s*and\s*Justification\s*of\s*the\s*Request(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*implementation\s*Schedule(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Description\s*of\s*(the)?\s*expected\s*outputs(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*Target\s*Area(,)?\s*Beneficiaries\s*and\s*Other\s*Stakeholders(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*Target\s*Area,\s*Beneficiaries\s*and\s*Population\s*Beneficiaries(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Anticipated\s*Economic\s*and\s*Social\s*Benefits(\s{3,}|\n)",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Lessons|LESSON)\s*(Learned|REFLECTED)(\b\w+\b\s*){0,5}",###### 2
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}(ESTIMATED|ESTIMATE|ESTIMATES)\s*(COSTS|COST)\s*(\b\w+\b\s*){0,4}",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}(COSTS|COST)\s*(ESTIMATED|ESTIMATE|ESTIMATES)(\b\w+\b\s*){0,4}",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*(COSTS|COST)(\b\w+\b\s*){0,4}",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}Expected\s*(Outputs|output|outcome|outcomes)(\b\w+\b\s*){0,4}",
                    r"((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(\b\w+\b\s*){0,4}(RESULT|RESULTS|LOGICAL|Result-based)\s*FRAMEWORK(\b\w+\b\s*){0,4}",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*cost\s*and\s*financing\s*arrangements", #MATCHES FOR A-B
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*Area\s*Beneficiaries",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|Projects)\s*Implementation",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Implementation",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Risks",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project|Projects|Project's)\s*(COST|COSTS)",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Key\s*performance\s*indicators",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Participatory\s*process\s*for\s*project\s*identification,\s*design\s*and\s*implementation",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Environmental\s*and\s*Social\s*Impact",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Sustainability|Governance|Risk\s*management|Knowledge\s*Building|Legal\s*instrument)",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*feasibility",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Implementation(\s{3,}|\n)",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Risks",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Technical\s*(Solutions|Solution)\s*Retained\s*and\s*Other\s*Alternatives",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Technical\s*(Solutions|Solution)\s*explored\s*and\s*Alternatives\s*Considered",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Loan\s*Conditions",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Policy\s*Dialogue",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Result-based\s*Framework",  
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Results-based\s*Framework",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Description\s*of\s*the\s*expected\s*outputs",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Cost\s*Estimates\s*of\s*the\s*project",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Outputs and Key activities ",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project’s Estimated Costs",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Technical\s*Solution\s*Adopted\s*and\s*Alternatives\s*Explored", 
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*ESTIMATED COSTS OF PREPARATORY ACTIVITIES",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*Cost\s*and\s*Financing\s*Arrangements",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Anticipated\s*Economic\s*and\s*Social\s*Benefits",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*implementation\s*arrangements", 
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Target\s*Beneficiaries",  # Matches "Target Beneficiaries"
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Background\s*and\s*Justification\s*of\s*the\s*Request",  # Matches "Background and Justification of the Request"       
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Institutional\s*and\s*Implementation\s*Arrangements ",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Expected\s*Project\s*Outcomes\s*and\s*Impacts",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*cost\s*and\s*funding\s*arrangements",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*cost\s*by\s*category\s*of\s*expenditures ",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Lessons\s*learned\s*related\s*to\s*efficiency",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project’s\s*Target\s*Area\s*and\s*Population\s*Beneficiaries",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Lessons\s*Reflected",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*cost\s*and\s*funding\s*arrangements",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Expected\s*Outputs",  # Matches "Expected Outputs"
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Programme\s*Beneficiaries",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*Target\s*Area,\s*Beneficiaries\s*and\s*Other\s*Stakeholders",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Technical\s*Solutions\s*Retained\s*and\s*Other\s*Alternatives\s*Explored",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Description\s*of\s*expected\s*outputs",  # Matches "Description of expected outputs"
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Description\s*of\s*the\s*expected\s*outputs",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Cost\s*Estimates\s*of\s*the\s*project",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Theory\*of\s*Change",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*IMPLEMENTATION,\s*MONITORING\s*AND\s*EVALUATION ",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Technical\s*Options\s*and\s*Alternative\s*Options\s*Considered",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Costing\s*and\s*Financial\s*Arrangements",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Implementation",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Budget/Cost\s*Structure",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Cost\s*Structure",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Budget",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Procurement(\s{3,}|\n)",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Procurement\s*Plan",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Procurement",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Conclusion\s*&\s*Recommendations",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Environmental\s*Impact",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Procurement\s*Plan",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Feasibility",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Implementation",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Budget/Cost\s*Structure",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Cost\s*Structure",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Budget",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Procurement(\s{3,}|\n)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Procurement",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Conclusion\s*&\s*Recommendations",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Sustainability|Governance|Environmental|Enivronment|Impact|Budget|Cost|Costs|Risk\s*management|Knowledge\s*Building|Legal\s*instrument)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Sustainability|Governance|Environmental|Enivronment|Impact|Budget|Cost|Costs|Risk\s*management|Knowledge\s*Building|Legal\s*instrument)",
                    r"SECTION 3: IMPLEMENTATION AND EVALUATION ",     
                    r"ESTIMATED\s*COSTS\s*AND\s*FINANCING\s*PLAN",
                    r"COST\s*ESTIMATES\s*AND\s*FINANCING\s*PLAN", 
                    r"LIST\s*OF\s*TABLES",  # Matches "LIST OF TABLES"
                    r"TABLES\s*OF\s*CONTENTS",  # Matches "TABLES OF CONTENTS"
                    r"LIST\s*OF\s*ACRONYMS",  # Matches "LIST OF ACRONYMS"
                    r"ACRONYMS",  # Matches "ACRONYMS"
                    r"RESULTS\s*FRAMEWORK\s*FOR\s*THE\s*IRSUE-LUXOR\s*PROGRAM",
                    r"IMPLEMENTATION AND EVALUATION",
                    r"RESULTS FRAMEWORK FOR THE IRSUE-LUXOR PROGRAM",
                    r"\n{1,}\s*RESULTS\s*FRAMEWORK",
                    r"RESULTS\s*FRAMEWORK(\s{3,}|\n)",
                    r"\n{1,}\s*RESULTS-FRAMEWORK",
                    r"RESULTS-FRAMEWORK(\s{3,}|\n)",
                    r"\n{1,}\s*LOGICAL-FRAMEWORK",
                    r"LOGICAL-FRAMEWORK(\s{3,}|\n)",
                    r"\n{1,}\s*LOGICAL\s*FRAMEWORK",
                    r"LOGICAL\s*FRAMEWORK(\s{3,}|\n)",
                    r"\n{1,}\s*RESULT\s*FRAMEWORK",
                    r"RESULT-BASED\s*FRAMEWORK(\s{3,}|\n)",
                    r"\n{1,}\s*RESULTS-BASED\s*FRAMEWORK",
                    r"RESULTS-BASED\s*FRAMEWORK(\s{3,}|\n)",
                    r"RESULTS\s*MATRIX(\s{3,}|\n)",
                    r"\n{1,}\s*RESULTS\s*MATRIX",
                    r"RESULT\s*FRAMEWORK(\s{3,}|\n)",
                    r"\n{1,}\s*RESULT\s*MATRIX(\s{3,}|\n)",
                    r"\n{1,}\s*POLICY\s*MATRIX",
                    r"POLICY\s*MATRIX(\s{3,}|\n)",
                    r"ANNEX\s*\d+\s*:\s*Procurement\s*Plan\s*\(.*?\)",
                    r"PARES II Programme Outputs and Expected Outcomes",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project|Program|Programm|Projects|Project's|Programme)\s*Outputs"
                    r"Programme Outputs and Expected Outcomes",
                    r"Program Outputs and Expected Results",
                    r"C\s{1}O\s{1}S\s{1}T",
                    r"P\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M",
                    r"T\s{1,}a\s{1,}b\s{1,}l\s{1,}e",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Project\s*cost\s*and\s*financing\s*arrangements",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*PROJECT\s*TARGET\s*AREA",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*BANK\s*EXPERIENCE\s*AND",
                    
                ]

                # Initialize a variable to store the match result
                components_text = None

                # Loop through the patterns to find matches
                for pattern in project_component_start:
                    try:
                        # Set the timeout for 5 seconds
                        signal.signal(signal.SIGALRM, timeout_handler)
                        signal.alarm(5)

                        # Attempt to find a match
                        match = re.search(pattern, text, re.IGNORECASE)
                        signal.alarm(0)  # Reset the alarm after the search

                        if match:
                            components_text = match.group(0).strip()
                            component_pattern = pattern
                            print(f"---------\n\nMatch found with pattern:----------- {pattern}\n\n")
                            components_text = clean_text(components_text, project_component_end)
                            components_text = re.sub(r'\n{3,}', '\n\n', components_text)
                            #print(components_text)# Exit the loop if a match is found
                            break
                    except TimeoutException:
                        print(f"Pattern took too long: {pattern}")
                        continue
                    except Exception as e:
                        print(f"An error occurred: {e}")
                        continue
                    finally:
                        signal.alarm(0)#ensure the alarm is reset even if an exception occurs


                # Fallback if no component text is found
                if not components_text:
                    components_text = "Project Components not found"
                    component_pattern = "Components Pattern N/A"

                extracted_text, summary_info = extract_summary_section(file_path)
                
                if not extracted_text:
                    extracted_text, summary_info = extract_description(file_path)
                    if extracted_text is None:
                        extracted_text = "Summary/Description not found"
                        summary_info = "Summary/Description pattern N/A"
                
                results.append([filename, extracted_text, summary_info, components_text, component_pattern])

            except Exception as e:
                results.append([filename, f"Error processing file: {e}", "Not Available", "Not Available", "Not Available"])

    df = pd.DataFrame(results, columns=["Filename", "Project Summary", "Summary Pattern", "Project Component", "Component Pattern"])
    df = remove_illegal_characters(df)  # Clean the DataFrame
    df.to_excel(output_excel, index=False)
    print(f"\n\nResults saved to {output_excel}")

folder_path = ""
output_excel = ""
process_pdfs_in_folder(folder_path, start_patterns, stop_patterns, output_excel)



Processing file: tanzania_-_africa_franchising_accelerator_project_-_technical_assistance_request.pdf
Table of Contents identified on page: 3
---------

Match found with pattern:----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)


Pattern used to truncate text:

----------- ((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Key\s*performance\s*indicators
Pattern used to truncate text:

----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Institutional\s*and\s*Implementation\s*Arrangements 
Pattern used to truncate text:

----------- SECTION 3: IMPLEMENTATION AND EVALUATION 
Table of Contents identified on page: 2
Start Summary Table not found.
TOC identified on page 2
Description Start pattern matched: '\d{1}\s*[-)\.]\s*n?\s*(PROPOSED|The)\s*(operation|activity|Project’s|PROGRAM|projects|PROGRAMM|PROGRAMME|Project|STUDY)' on page 11
Stop pattern not found. Returning text after start pattern.

Process

Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b on page 5
Actual line containing start pattern: None
Matched end pattern: \n{1,}\s*Result on page Unknown
Summary Table found!!!

Processing file: multinational-desert_to_power_regional_technical_assistance_project_for_the_sahel_-_project_appraisal_report.pdf
Table of Contents identified on page: 10
---------

Match found with pattern:----------- \b[B-F]\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)


Pattern used to truncate text:

----------- \b[B-F]\s*[-\.)]?\s*\n?\s*Project\s*cost\s*and\s*financing\s*arrangements
Table of Contents identified on page: 9
Start Summary Table not found.
TOC identified on page 9
Description Start pattern matched: '(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(operation|activity|Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|projects|STUDY)\s*Description' on page 13
Description Stop 

Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b on page 6
Actual line containing start pattern: None
Matched end pattern: \n{1,}\s*Result on page 3
Summary Table found!!!

Processing file: multinational_-_330_kv_wapp_ghana_-_burkina_-_mali_interconnection_project_-_project_information_memorendum.pdf
Table of Contents identified on page: 5
---------

Match found with pattern:----------- (Project|Programme|Programm|Program)\s*Components([\s\S]*)


Pattern used to truncate text:

----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Environmental\s*and\s*Social\s*Impact
Pattern used to truncate text:

----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Feasibility
Table of Contents identified on page: 4
Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b on page 2
Actual line containing start pattern: None
Matched 

Table of Contents identified on page: 3
---------

Match found with pattern:----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)


Pattern used to truncate text:

----------- \b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*feasibility
Pattern used to truncate text:

----------- ((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Bank\s*Group\s*experience,\s*lessons reflected in project design
Pattern used to truncate text:

----------- ((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME)\s*(Estimated\s*Costs)|(Target\s*Area\s*and)
Pattern used to truncate text:

----------- ((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*(PROJECT|PROGRAM|PROGRAMM|PROGRAMME)\s*TYPE(\s{3,}|\n)
Table of Contents identified on page: 2
Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|

Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b on page 6
Actual line containing start pattern: None
Matched end pattern: \n{1,}\s*Results-based on page 8
Summary Table found!!!

Processing file: sao_tome_-_etudes_techniques_complementaires_et_de_structuration_financiere_du_projet.pdf

Processing file: proposal_for_a_grant_of_us_500000_humanitarian_emergency_assistance_to_overcome_the_oil_spill_crisis_in_mauritius.pdf
Table of Contents identified on page: 5
Table of Contents identified on page: 4
Start Summary Table not found.
TOC identified on page 4
Start pattern not found.

Processing file: djibouti_-_geothermal_exploration_project_in_the_lake_assal_region_-_memorandum.pdf
Table of Contents identified on page: 4
---------

Match found with pattern:----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)


Pattern used to truncate 

...

Table of Contents identified on page: 29
Table of Contents identified on page: 28
Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b on page 8
Actual line containing start pattern: None
Matched end pattern: \n{1,}\s*Results-based on page 10
Summary Table found!!!

Processing file: drc_-_public_finance_modernization_support_project_pam-fp_-_project_appraisal_report_0.pdf
Table of Contents identified on page: 3
---------

Match found with pattern:----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Objectives\s*and\s*Components([\s\S]*)


Pattern used to truncate text:

----------- \b[IVIX]{1,3}[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project)\s*feasibility
Pattern used to truncate text:

----------- \b[IVIX]{1,3}[-\.)]?\s*\n?\s*Implementation
Table of Contents identified on page: 2
Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUM

Table of Contents identified on page: 5
---------

Match found with pattern:----------- \d{1}\s*[-)\.]\s*n?\s*(main|overall)?\s*(Project|Program|Programme|Programm|project's)?\s*Components([\s\S]*)


Pattern used to truncate text:

----------- \b[IVIX]{1,3}[-\.)]?\s*\n?\s*Implementation
Pattern used to truncate text:

----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Sustainability|Governance|Environmental|Enivronment|Impact|Budget|Cost|Costs|Risk\s*management|Knowledge\s*Building|Legal\s*instrument)
Table of Contents identified on page: 4
Start Summary Table not found.
TOC identified on page 4
Description Start pattern matched: '(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(PROPOSED|The)\s*(operation|activity|Project’s|PROGRAM|projects|PROGRAMM|PROGRAMME|Project|STUDY)' on page 8
Description Stop pattern matched: '(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|OPERATIONAL|OPERATION)\s*IMPLEMENTATION' on page 15

Processing f

Table of Contents identified on page: 4
Matched start pattern: \n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b on page 6
Actual line containing start pattern: None
Matched end pattern: \n{1,}\s*Result on page 8
Summary Table found!!!

Processing file: multinational_-_risking_agricultural_finance_for_smallholder_farmers_-_technical_assistance_request.pdf
Table of Contents identified on page: 2
---------

Match found with pattern:----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT)\s*Components([\s\S]*)


Pattern used to truncate text:

----------- ((\b[B-F])|(\b\d{1,2}(\.\d{1,2}){0,2}))\s*[-\.)]?\s*\n?\s*Key\s*performance\s*indicators
Pattern used to truncate text:

----------- (\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*Implementation
Table of Contents identified on page: 1
Start Summary Table not found.
TOC identified on page 1
Description Start pattern matched: '(\b\d{1

/var/folders/pd/vl7v8b496h3fs87b4fnt9qcw0000gn/T/ipykernel_38212/1180633994.py:36: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(clean_cell)
/var/folders/pd/vl7v8b496h3fs87b4fnt9qcw0000gn/T/ipykernel_38212/1180633994.py:586: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df.to_excel(output_excel, index=False)




Results saved to /Users/adilqasin/Documents/WBG/Scrapping/AFDB/Outputs/13_Dec.xlsx
